In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader


In [ ]:
# Write your code here

transform_train = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
transform_test = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

train_dir = os.path.join(path, "PlantVillage", "test")
test_dir = os.path.join(path, "PlantVillage", "train")

# OR


from torchvision.datasets import ImageFolder

# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir, transform=transform_train)
test_dataset  = ImageFolder(root=test_dir,  transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def denormalize(img):

  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = np.clip(img, 0, 1)
  return img

xxlable = {0:"Early_blight",1:"late_blight",2:"healthy_blight"}
fig, axes = plt.subplots(1, 4, figsize=(16, 8))
images, labels = next(iter(train_loader))
for i in range(4):
    # TODO: Get an image-mask pair from train_dataset
    # Hint: Use train_dataset[i] to get the i-th sample

   # Get a batch of training images

    print(f"Batch shape: {images.shape}, Labels: {labels}")

    # Display image (denormalize first)
    axes[i].imshow(denormalize(images[i]))
    axes[i].set_title(f"potato: {xxlable[labels[i].item()]}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    """U-Net model with skip connections (Residuals)."""
    def __init__(self,num_classes=10):
        super(CustomModel, self,).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2) #32-> 16

        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),


        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            nn.Linear(512 * 16 * 16, 3),
            # Softmax Layer
            nn.Softmax(dim=1)  # Apply along the class dimension
        )# Fully connected layer for classification (10 classes)

    def forward(self, x):
        # --- ENCODER ---
        x = self.conv1(x)

        x = self.conv2(x)

        x = self.conv3(x)

        x = self.conv4(x)

        x = self.conv5(x)

        x = self.classifier(x)
        return x



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:

# Training and Validation Loops
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

def validate(model, dataloader, criterion, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total  # Return accuracy

In [ ]:
# Write your code here


# Run Training
model = CustomModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):  # Train for 5 epochs
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    accuracy = validate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Validation Accuracy = {accuracy:.2f}%")



In [ ]:
# Write your code here
